# 支持向量机（SVM）

`sklearn.svm.SVC` 在拉平后的 `784` 维特征上训练。  
原仓库 `max_iter=10` 几乎不迭代；这里改为较大上限以便得到合理结果（仍可能较慢）。


In [1]:
from pathlib import Path

import numpy as np
import torchvision
from sklearn.metrics import confusion_matrix
from sklearn import svm


## 1. 超参数与样本量


In [2]:
TRAIN_N = 60000  # 可改小如 10000 加快实验
TEST_N = 2000

# 与 svm.py 接近；max_iter 提高以适配新版 sklearn
SVM_C = 5.0
SVM_GAMMA = 0.05
SVM_MAX_ITER = -1  # -1：不限制迭代（全量很慢时可减小 TRAIN_N）


## 2. 读入数据并归一化到 [0,1]


In [3]:
from pathlib import Path

import torchvision
from mnist_from_raw import load_all_numpy, raw_files_available

if raw_files_available():
    tr_x, tr_y, te_x, te_y = load_all_numpy()
    train_x = tr_x[:TRAIN_N].astype(np.float64) / 255.0
    train_y = tr_y[:TRAIN_N]
    test_x = te_x[:TEST_N].astype(np.float64) / 255.0
    test_y = te_y[:TEST_N]
    print("数据来源: data/raw", "train", train_x.shape, "test", test_x.shape)
else:
    MNIST_ROOT = Path("./mnist")
    download = not MNIST_ROOT.is_dir() or not any(MNIST_ROOT.iterdir())

    def load_train(n, download):
        ds = torchvision.datasets.MNIST(
            root=str(MNIST_ROOT),
            train=True,
            transform=torchvision.transforms.ToTensor(),
            download=download,
        )
        x = ds.data.numpy()[:n].astype(np.float64) / 255.0
        y = ds.targets.numpy()[:n]
        return x, y

    def load_test(n, download):
        ds = torchvision.datasets.MNIST(
            root=str(MNIST_ROOT),
            train=False,
            transform=torchvision.transforms.ToTensor(),
            download=download,
        )
        x = ds.data.numpy()[:n].astype(np.float64) / 255.0
        y = ds.targets.numpy()[:n]
        return x, y

    train_x, train_y = load_train(TRAIN_N, download)
    test_x, test_y = load_test(TEST_N, download)
    print("数据来源: torchvision", "train", train_x.shape, "test", test_x.shape)


数据来源: data/raw train (60000, 28, 28) test (2000, 28, 28)


## 3. 特征：展平为二维矩阵 `(N, 784)`


In [4]:
X_train = train_x.reshape(train_x.shape[0], -1)
X_test = test_x.reshape(test_x.shape[0], -1)


## 4. 训练 SVM


In [5]:
clf = svm.SVC(C=SVM_C, gamma=SVM_GAMMA, max_iter=SVM_MAX_ITER)
clf.fit(X_train, train_y)
print("support vectors:", clf.n_support_)


support vectors: [1790  658 3001 2622 2280 2720 1871 1894 3008 2489]


## 5. 测试集准确率与混淆矩阵


In [6]:
test_pred = clf.predict(X_test)
test_acc = (test_pred == test_y).mean()
print("test accuracy:", test_acc)

cm = confusion_matrix(test_y, test_pred)
print("confusion matrix shape:", cm.shape)


test accuracy: 0.9785
confusion matrix shape: (10, 10)


## 6.（可选）训练集上的拟合情况


In [7]:
train_pred = clf.predict(X_train)
train_acc = (train_pred == train_y).mean()
print("train accuracy:", train_acc)


train accuracy: 1.0
